<a href="https://colab.research.google.com/github/ojos168/Algorithmic_Empathy/blob/main/Stress_Prediction_EDA_and_Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Data Loading and Setup

In this section, I load the required Python libraries for data manipulation and visualization. I also load the raw dataset from a CSV file. The dataset used is the Human Stress Prediction Dataset, containing Reddit posts labeled as 1 (stress) or 0 (no stress).

In [ ]:
# Install SHAP for explainable AI and Gensim for Word2Vec
!pip install shap gensim -q

In [ ]:
# import necessary libraries for data science and text processing
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import CountVectorizer

# set the visual style for academic charts
sns.set_theme(style="whitegrid", palette="pastel")

# load the dataset
url = 'https://raw.githubusercontent.com/ojos168/HW2_Stress_Prediction/refs/heads/main/Stress.csv'
df = pd.read_csv(url)

# display basic information about the dataset
print("Dataset Shape (Rows, Columns):", df.shape)
display(df.head())

In [ ]:
# Import Explainability and Semantic Alignment Libraries
import shap
import gensim.downloader as api
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Initialize SHAP for notebook visualizations
shap.initjs()

##Data Cleaning and Feature Engineering

The text need to be clean before analyzed. I remove duplicates, missing values, URLs, and punctuation to reduce noise. Furthermore, I engineer two new features to help answer my research question:
- word_count: The total number of words in a post.
- first_person_count: The frequency of first-person pronouns (I, me, my), which psycholinguistic theory suggests is an indicator of psychological distress.

In [ ]:
# step 1: drop missing values and duplicates to ensure data quality
df = df.dropna(subset=['text', 'label'])
df = df.drop_duplicates(subset=['text'])

# step 2: define a function to clean the raw Reddit text
def clean_text(text):
  text = str(text).lower() # Convert to lowercase
  text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
  text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
  return text

# apply the cleaning function to create a new column
df['cleaned_text'] = df['text'].apply(clean_text)

# step 3: Feature Engineering - calculate Word Count
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))

# step 4: feature engineering - count first-person pronouns
# i use regex to find standalone words: i, me, my, mine, myself
def count_first_person(text):
  pronouns = re.findall(r'\b(i|me|my|mine|myself)\b', text)
  return len(pronouns)

df['first_person_count'] = df['cleaned_text'].apply(count_first_person)

print(f"Data cleaning finished. Remaining rows: {df.shape[0]}")
display(df[['cleaned_text', 'label', 'word_count', 'first_person_count']].head())

##Exploratory Data Analysis (EDA)

In this section, I create four visualizations to understand the distributions within my dataset and explore preliminary relationships between language use and stress labels.

In [ ]:
# create a figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Exploratory Data Analysis: Linguistic Markers of Stress', fontsize=18, fontweight='bold', y=1.02)

# visualization 1: distribution of labels (target variable)
# connection to RQ: ensures the dataset is balanced for training my NLP model.
sns.countplot(data=df, x='label', ax=axes[0, 0], palette=['#64b5f6', '#e57373'])
axes[0, 0].set_title('Fig 2: Class Distribution (Stress vs. Non-Stress)')
axes[0, 0].set_xticklabels(['No Stress (0)', 'Stress (1)'])
axes[0, 0].set_xlabel('Psychological State')
axes[0, 0].set_ylabel('Number of Reddit Posts')

# visualization 2: Word Count by Class
# connection to RQ: checks if stressed users write longer posts (a potential behavioral marker).
sns.boxplot(data=df, x='label', y='word_count', ax=axes[0, 1], palette=['#64b5f6', '#e57373'])
axes[0, 1].set_title('Fig 3: Post Length (Word Count) Distribution')
axes[0, 1].set_xticklabels(['No Stress (0)', 'Stress (1)'])
axes[0, 1].set_xlabel('Psychological State')
axes[0, 1].set_ylabel('Total Words per Post')
# limit y-axis to remove extreme outliers and make the boxplot readable
axes[0, 1].set_ylim(0, df['word_count'].quantile(0.95))



# visualization 3: first-person pronoun frequency
# connection to RQ: directly tests a psycholinguistic marker (LIWC) before complex ML modeling.
sns.violinplot(data=df, x='label', y='first_person_count', ax=axes[1, 0], palette=['#64b5f6', '#e57373'])
axes[1, 0].set_title('Fig 4: First-Person Pronoun Usage by Class')
axes[1, 0].set_xticklabels(['No Stress (0)', 'Stress (1)'])
axes[1, 0].set_xlabel('Psychological State')
axes[1, 0].set_ylabel('Count of "I, me, my"')
axes[1, 0].set_ylim(0, 30)

# visualization 4: Top 10 Most Frequent Words in Stress Posts
# connection to RQ: shows the semantic content of stress posts to guide my future SHAP explanation analysis.
stress_texts = df[df['label'] == 1]['cleaned_text']
# I use CountVectorizer to count words, ignoring standard english stop words
vectorizer = CountVectorizer(stop_words='english', max_features=10)
word_matrix = vectorizer.fit_transform(stress_texts)
word_freq = pd.DataFrame(word_matrix.sum(axis=0), columns=vectorizer.get_feature_names_out()).T
word_freq.columns = ['Frequency']
word_freq = word_freq.sort_values(by='Frequency', ascending=False)

sns.barplot(x=word_freq['Frequency'], y=word_freq.index, ax=axes[1, 1], color='#e57373')
axes[1, 1].set_title('Fig 5: Top 10 High-Frequency Words in Stress Posts')
axes[1, 1].set_xlabel('Total Frequency')
axes[1, 1].set_ylabel('Keywords (Stopwords removed)')

plt.tight_layout()
plt.show()


##Machine Learning and Explainable AI (SHAP)

In [ ]:
# install the SHAP library for explainable AI
!pip install shap

###TF-IDF & Model Training

In [ ]:
# import machine learning libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# step 1: split the dataset into 80% training and 20% testing sets
# I use 'stratify=y' to ensure the balance of Stress/Non-Stress is maintained
X = df['cleaned_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# step 2: feature vectorization using TF-IDF
# I limit to the top 1000 most informative words to prevent noise and speed up SHAP
vectorizer = TfidfVectorizer(max_features=1000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# step 3: train and evaluate Logistic Regression
print("-----Logistic Regression Performance-----")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
print(classification_report(y_test, lr_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, lr_model.predict_proba(X_test_tfidf)[:, 1]):.4f}\n")

# step 4: train and evaluate Random Forest
print("-----Random Forest Performance-----")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)
rf_preds = rf_model.predict(X_test_tfidf)
print(classification_report(y_test, rf_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test_tfidf)[:, 1]):.4f}\n")

##Comprehensive Model Evaluation

In [ ]:
#step5: compenhensive model training
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

def evaluate_and_print_metrics(model, X_test_features, y_test_labels, model_name):
    """
    Evaluates a trained machine learning model and prints standard classification metrics.
    """
    print(f"========== {model_name} Performance ==========")

    y_pred = model.predict(X_test_features)
    y_prob = model.predict_proba(X_test_features)[:, 1]

    accuracy = accuracy_score(y_test_labels, y_pred)
    precision = precision_score(y_test_labels, y_pred)
    recall = recall_score(y_test_labels, y_pred)
    f1 = f1_score(y_test_labels, y_pred)
    roc_auc = roc_auc_score(y_test_labels, y_prob)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}\n")

    # Plot Confusion Matrix
    cm = confusion_matrix(y_test_labels, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Stress', 'Stress'],
                yticklabels=['Non-Stress', 'Stress'])
    plt.title(f'Confusion Matrix: {model_name}', fontsize=12, fontweight='bold')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Run evaluation for both models
evaluate_and_print_metrics(lr_model, X_test_tfidf, y_test, "Logistic Regression")
evaluate_and_print_metrics(rf_model, X_test_tfidf, y_test, "Random Forest")

##SHAP Explainability Analysis


In [ ]:
import shap
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

# Initialize SHAP visualization
shap.initjs()

print("Generating SHAP Explanations...")
# Use LinearExplainer for Logistic Regression
explainer = shap.LinearExplainer(lr_model, X_train_tfidf, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_test_tfidf)

# Plot SHAP Summary
plt.figure(figsize=(10, 6))
plt.title("SHAP Feature Importance for Stress Prediction", fontsize=14, fontweight='bold')
shap.summary_plot(shap_values, X_test_tfidf, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()


##Senmatic Alignment
I use Word2Vec to mathematically prove that the AI's top words match human psychology (LIWC)

In [ ]:
# algorithm empathy

# 1. Extract the top 20 words prioritized by the LR model
coefficients = lr_model.coef_[0]
top_20_indices = np.argsort(coefficients)[-20:]
top_machine_words = [feature_names[i] for i in top_20_indices]

print("\nTop 20 words identified by the Machine (SHAP/LR):")
print(top_machine_words)
print("-" * 50)

# 2. Define human psycholinguistic markers (Simulated LIWC)
liwc_human_markers = ['i', 'me', 'my', 'anxious', 'sad', 'angry', 'depressed',
                      'fear', 'nervous', 'cry', 'stress', 'alone', 'overwhelmed']

# 3. Load Word2Vec model
print("Loading Google News Word2Vec model (Please wait)...")
w2v_model = api.load("word2vec-google-news-300")
print("Word2Vec model loaded successfully!")

# 4. Calculate Cosine Similarity
def calculate_semantic_alignment(machine_words, human_words, w2v):
    valid_machine = [w for w in machine_words if w in w2v]
    valid_human = [w for w in human_words if w in w2v]

    if not valid_machine or not valid_human:
        return 0.0

    machine_vectors = np.array([w2v[w] for w in valid_machine])
    human_vectors = np.array([w2v[w] for w in valid_human])

    machine_centroid = np.mean(machine_vectors, axis=0).reshape(1, -1)
    human_centroid = np.mean(human_vectors, axis=0).reshape(1, -1)

    alignment_score = cosine_similarity(machine_centroid, human_centroid)[0][0]
    return alignment_score

final_alignment_score = calculate_semantic_alignment(top_machine_words, liwc_human_markers, w2v_model)

print("\n" + "="*50)
print(f"Algorithm Empathy Score: {final_alignment_score:.4f}")
print("="*50)
print("Note: A higher score (closer to 1.0) indicates stronger semantic alignment with human psychology.")